## Measuring Performances

This notebook computes 31 verification metrics for bias-corrected precipitation products. It compares each correction method (LS, LSEQM, LSEQM+DL) against three reference datasets (CPC, IMERGL, IMERGF) for a user-selected month and dekad.

### Metrics computed (31 total)

| Category | Metrics |
|----------|--------|
| **Continuous** | Relative Bias, Pearson Correlation, RMSE, MAE, NSE, Std Dev (ref/test), Std Dev Ratio, KS-test (stat + p-value) |
| **Categorical** (1 mm/day threshold) | POD, FAR, CSI, FPD (ref/test), MDWP (ref/test), Dry Spell Length (ref/test) |
| **Percentiles** | p25, p50, p75, p90, p95, p99 (ref and test) |

### Two output modes

- **Timeseries** (Step 3): One metric grid per year, dims = (time, lat, lon).
- **Single dekad** (Step 4): All years pooled, dims = (lat, lon).

### Perfect scores

| Metric | Perfect | Good threshold |
|--------|---------|----------------|
| RB     | 0       | within +/- 0.25 |
| CORR   | 1       | > 0.7 |
| RMSE, MAE | 0   | -- |
| NSE    | 1       | > 0.5 |
| POD    | 1       | > 0.6 |
| FAR    | 0       | < 0.3 |
| CSI    | 1       | > 0.5 |

In [ ]:
# ==============================================================
# Step 1: Environment Setup
# ==============================================================

import os
import sys
import logging

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import initialize_config
import src.config as config

initialize_config(os.path.join(project_root, 'config.yml'))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()],
    force=True
)

print(f"Project root: {project_root}")
print(f"CPC file:     {config.cpc_file}")
print(f"IMERGL file:  {config.imergl_file}")
print(f"IMERGF file:  {config.imergf_file}")
print(f"Metrics path: {config.metrics_path_template}")

### Step 2: User Inputs

Select the month (1--12) and dekad (1, 2, or 3) for which to compute metrics.

In [ ]:
# ==============================================================
# Step 2: Gather User Inputs
# ==============================================================

month_input = input('Enter the month (1-12): ').strip()
dekad_input = input('Enter the dekad (1, 2, or 3): ').strip()

try:
    month = int(month_input)
    assert 1 <= month <= 12
except (ValueError, AssertionError):
    raise SystemExit('Invalid month. Please provide a number from 1 to 12.')

try:
    dekad = int(dekad_input)
    assert dekad in (1, 2, 3)
except (ValueError, AssertionError):
    raise SystemExit('Invalid dekad. Must be 1, 2, or 3.')

print(f'\nSelected: month={month}, dekad={dekad}')

### Step 3: Timeseries Metrics (per-year)

For each year present in both reference and test data, compute all 31 metrics across the ~10-day dekad window. This produces one metric grid per year, allowing you to track correction quality over time.

Nine combinations are computed: {CPC, IMERGL, IMERGF} x {LS, LSEQM, LSEQM+DL}.

In [ ]:
# ==============================================================
# Step 3: Compute Timeseries Metrics
# ==============================================================

from src.metrics import run_metrics_pipeline

ts_files = run_metrics_pipeline(month, dekad, mode='timeseries')

print(f'\nTimeseries output files:')
for f in ts_files:
    if f is not None:
        print(f'  {os.path.basename(f)}')

### Step 4: Single Dekad Metrics (aggregated)

Pool all years together and compute one summary metric grid per combination. This collapses the temporal dimension, giving a spatial overview of correction quality across the entire record.

In [ ]:
# ==============================================================
# Step 4: Compute Single Dekad Metrics
# ==============================================================

sd_files = run_metrics_pipeline(month, dekad, mode='single')

print(f'\nSingle dekad output files:')
for f in sd_files:
    if f is not None:
        print(f'  {os.path.basename(f)}')

### Step 5: Inspect Results

Load one of the output files and visualize a sample metric to verify correctness.

In [ ]:
# ==============================================================
# Step 5: Inspect a Sample Result
# ==============================================================

import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# Pick the first available single-dekad file (prefer CPC vs LSEQMDL)
sample_file = None
for f in sd_files:
    if f is not None and os.path.isfile(f):
        sample_file = f
        if 'lseqmdl' in f:
            break

if sample_file is not None:
    ds = xr.open_dataset(sample_file, engine=config.NETCDF_ENGINE)
    print(f'File: {os.path.basename(sample_file)}')
    print(f'Variables ({len(ds.data_vars)}): {list(ds.data_vars)}')
    print(f'Dimensions: {dict(ds.dims)}')

    # Plot NSE spatial map
    if 'nse' in ds.data_vars:
        fig, ax = plt.subplots(figsize=(10, 5))
        nse = ds['nse']
        if 'time' in nse.dims:
            nse = nse.isel(time=-1)
        im = ax.pcolormesh(
            nse.lon, nse.lat, nse.values,
            cmap='RdYlGn', vmin=-1, vmax=1, shading='auto'
        )
        ax.set_xlim(95, 141)
        ax.set_ylim(-11, 6)
        ax.set_title(f'NSE -- {os.path.basename(sample_file)}')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_aspect('equal')
        fig.colorbar(im, ax=ax, label='NSE')
        plt.tight_layout()
        plt.show()

        vals = nse.values[~np.isnan(nse.values)]
        if len(vals) > 0:
            print(f'NSE: mean={np.mean(vals):.4f}, '
                  f'median={np.median(vals):.4f}, '
                  f'% > 0.5: {100 * np.mean(vals > 0.5):.1f}%')

    ds.close()
else:
    print('No output files found. Check that corrected precipitation files exist.')

### Step 6: Metric Interpretation Guide

Quick reference for interpreting the 31 metrics.

In [ ]:
# ==============================================================
# Step 6: Print Metric Interpretation Reference
# ==============================================================

interpretation = [
    ('Lower is better', 'RMSE, MAE, FAR, |RB|, KS stat'),
    ('Higher is better', 'CORR, NSE, POD, CSI, KS p-value'),
    ('Target similarity', 'Percentiles, STDEV, MDWP, FPD, DSL'),
]

print('Metric Interpretation Guide')
print('=' * 55)
for direction, metrics in interpretation:
    print(f'  {direction:20s}  {metrics}')

print()
print('Units')
print('-' * 55)
print('  mm/day:      RMSE, MAE, STDEV, MDWP, Percentiles')
print('  unitless:    RB, CORR, NSE, POD, FAR, CSI, KS')
print('  percent:     FPD')
print('  days:        DSL')

### Summary

This notebook computed 31 verification metrics for 9 reference-vs-test combinations in two modes (timeseries and single dekad). The output NetCDF files are stored in the method-specific metrics directories:

- `data/output/metrics_ls/`
- `data/output/metrics_lseqm/`
- `data/output/metrics_lseqmdl/`

These files feed into notebook `04_qa_framework` for composite quality assessment.